# x402 `batch-settlement` with Permit2 — EURe facilitator test

The EURe counterpart of `x402_batch_settlement.ipynb`, kept short:

1. the buyer opens a channel with an EURe **deposit**, paid via Permit2;
2. several requests accumulate as off-chain **vouchers**;
3. the seller **claims** them in one transaction and pays the 0.01 EURe fee.

**The difference from USDC/EURC is the deposit.** EURe has no EIP-3009, so the buyer signs a Permit2
transfer to the batch-settlement **Permit2 deposit collector** (`0x4020…0005`), whose witness is the
channel id. An EIP-2612 permit for Permit2 travels with it (`eip2612GasSponsoring`), so the buyer
needs no ETH and no prior `approve()`. Vouchers and claims are the same as for any token.

**Prerequisites**:
- EURe on the buyer. On Base Sepolia it comes from the Monerium sandbox; see `README.md` →
  *Test EURe*.
- A local facilitator built from the Permit2 branch: `npm run build && npm run dev`.
- The seller needs a little ETH for its one EURe fee `approve()`.

`DRY_RUN=1` skips every on-chain write (deposit, approve, claim). The off-chain parts and
`/verify` still run.

In [1]:
// Setup: imports, accounts, network
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount, generatePrivateKey } from "npm:viem@2/accounts";
import { createPublicClient, createWalletClient, http, formatUnits, parseUnits, getAddress } from "npm:viem@2";
import { base, baseSepolia } from "npm:viem@2/chains";

const env = await load({ envPath: "../.env", examplePath: null, export: true });
const toKey = (k: string) => `0x${k.replace(/^0x/, "")}` as `0x${string}`;
// Same buyer selection as x402_exact_permit2.ipynb: the address linked in the Monerium sandbox.
const BUYER_KEY_VAR = env.EURE_BUYER_PRIVATE_KEY ? "EURE_BUYER_PRIVATE_KEY" : "TEST_WALLET_PRIVATE_KEY";
const buyer = privateKeyToAccount(toKey(env[BUYER_KEY_VAR]));
const seller = privateKeyToAccount(toKey(env.NFT_WALLET_PRIVATE_KEY));
// The seller's claim-signing key. In production the seller keeps it; here it is throwaway.
const receiverAuthorizer = privateKeyToAccount(generatePrivateKey());

const USE_MAINNET = false;            // ⚠️ true = REAL MONEY
const CONFIRM_MAINNET_WRITES = false; // second opt-in for the on-chain writes on mainnet
const DRY_RUN = Deno.env.get("DRY_RUN") === "1";
const mayWrite = !DRY_RUN && (!USE_MAINNET || CONFIRM_MAINNET_WRITES);

const net = USE_MAINNET
    ? { chain: base, caip2: "eip155:8453", eure: "0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C", explorer: "https://basescan.org/tx/" }
    : { chain: baseSepolia, caip2: "eip155:84532", eure: "0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85", explorer: "https://sepolia.basescan.org/tx/" };
const EURE = net.eure as `0x${string}`;
const fmt = (x: bigint | string) => `${formatUnits(BigInt(x), 18)} EURe`; // EURe has 18 decimals
const rpcUrl = env[`RPC_URL_EIP155_${net.caip2.split(":")[1]}`] ?? net.chain.rpcUrls.default.http[0];
const publicClient = createPublicClient({ chain: net.chain, transport: http(rpcUrl) });

const FACILITATOR_URL = "http://localhost:8080";
const post = async (path: string, body: unknown) =>
    (await fetch(`${FACILITATOR_URL}${path}`, {
        method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(body),
    })).json();

console.log(`🧪 ${net.chain.name} (${net.caip2})${DRY_RUN ? "  — DRY_RUN: no on-chain writes" : ""}`);
console.log(`   Buyer:  ${buyer.address}  (from ${BUYER_KEY_VAR})`);
console.log(`   Seller: ${seller.address}`);

🧪 Base (eip155:8453)
   Buyer:  0x553179556FC2A39e535D65b921e01fA995E79101  (from TEST_WALLET_PRIVATE_KEY)
   Seller: 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## 1. `/supported`

The facilitator must offer `batch-settlement` on this network and list EURe's fee in 18 decimals.

In [2]:
const supported = await (await fetch(`${FACILITATOR_URL}/supported`)).json();
const hasBatch = supported.kinds.some((k: any) => k.scheme === "batch-settlement" && k.network === net.caip2);
const eureFee = supported.facilitatorFees?.assets?.find((a: any) => a.network === net.caip2 && a.symbol === "EURe");
if (!hasBatch || !eureFee) {
    throw new Error("No batch-settlement or no EURe fee asset here — run a local facilitator from the Permit2 branch.");
}
const facilitator = supported.facilitatorFees.recipient as `0x${string}`;
const FEE = BigInt(eureFee.flatFee);
console.log(`✅ batch-settlement on ${net.caip2}; claim fee ${fmt(FEE)}; facilitator ${facilitator}`);

✅ batch-settlement on eip155:8453; claim fee 0.01 EURe; facilitator 0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206


## 2. Buyer opens a channel: Permit2 deposit + first voucher

`assetTransferMethod: "permit2"` makes the SDK client sign the deposit through Permit2. Because
the 402 declares `eip2612GasSponsoring`, the client also signs the EIP-2612 permit; it reads the
buyer's Permit2 allowance and EURe `nonces` through `readContract` to decide whether to.

In [3]:
import { x402Client } from "npm:@x402/fetch@^2.26.0";
import { BatchSettlementEvmScheme, InMemoryClientChannelStorage, signVoucher }
    from "npm:@x402/evm@^2.26.0/batch-settlement/client";

const buyerSigner = {
    address: buyer.address,
    signTypedData: (msg: any) => buyer.signTypedData(msg),
    readContract: (args: any) => publicClient.readContract(args),
};
const client = new x402Client();
client.register(net.caip2 as any, new BatchSettlementEvmScheme(buyerSigner as any, { storage: new InMemoryClientChannelStorage() }));
client.setSpendControls({ allowedAssets: [{ network: net.caip2 as any, asset: EURE }] });

const PRICE = parseUnits("0.004", 18); // per request; the deposit is 5× this
const paymentRequirements = {
    scheme: "batch-settlement",
    network: net.caip2,
    amount: PRICE.toString(),
    asset: EURE,
    payTo: seller.address,
    maxTimeoutSeconds: 3600,
    extra: {
        name: "Monerium EURe", version: "1", // EURe's EIP-712 domain, for the EIP-2612 permit
        assetTransferMethod: "permit2",
        receiverAuthorizer: receiverAuthorizer.address,
        withdrawDelay: 86400,
    },
};
const depositPayload: any = await client.createPaymentPayload({
    x402Version: 2,
    accepts: [paymentRequirements],
    resource: { url: "https://example.com/llm", description: "EURe batch test", mimeType: "application/json" },
    extensions: { eip2612GasSponsoring: {} },
} as any);

const p2 = depositPayload.payload.deposit.authorization.permit2Authorization;
console.log(`✍️  deposit ${fmt(depositPayload.payload.deposit.amount)} via Permit2`);
console.log(`   spender:   ${p2.spender}  (batch Permit2 deposit collector)`);
console.log(`   channelId: ${p2.witness.channelId}`);
console.log(`   voucher #1 cumulative: ${fmt(depositPayload.payload.voucher.maxClaimableAmount)}`);
console.log(depositPayload.extensions?.eip2612GasSponsoring
    ? "🔏 EIP-2612 permit for Permit2 attached"
    : "ℹ️  no permit — the buyer already approved Permit2 enough");

✍️  deposit 0.02 EURe via Permit2
   spender:   0x4020425FAf3B746C082C2f942b4E5159887B0005  (batch Permit2 deposit collector)
   channelId: 0x667e3f8fa3eb486bf1ee5e3b3ebd65c15ba04e1611cec09f8db1e475af0785c0
   voucher #1 cumulative: 0.004 EURe
🔏 EIP-2612 permit for Permit2 attached


## 3. `/verify` and `/settle` the deposit

The deposit is free (no facilitator fee). With an unfunded buyer, `/verify` fails on the balance.

In [4]:
const verify = await post("/verify", { paymentPayload: depositPayload, paymentRequirements });
console.log("verify:", JSON.stringify(verify));

let depositSettled = false;
if (!verify.isValid) {
    console.log("⏭️  deposit not settled — /verify did not pass");
} else if (!mayWrite) {
    console.log("⏭️  deposit not settled (DRY_RUN or unconfirmed mainnet)");
} else {
    const settled = await post("/settle", { paymentPayload: depositPayload, paymentRequirements });
    depositSettled = settled.success;
    console.log(settled.success ? `✅ channel funded: ${net.explorer}${settled.transaction}` : `❌ ${JSON.stringify(settled)}`);
}

verify: {"isValid":true,"payer":"0x553179556FC2A39e535D65b921e01fA995E79101","extra":{"channelId":"0x667e3f8fa3eb486bf1ee5e3b3ebd65c15ba04e1611cec09f8db1e475af0785c0","balance":"0","totalClaimed":"0","withdrawRequestedAt":0,"refundNonce":"0"}}
✅ channel funded: https://basescan.org/tx/0x76965b4bd7c651b20476c9e5725eb9dd2515af3df6d544cb5ff0320a7c4c1fe8


## 4. Two more requests, off-chain

Each request is just a new voucher with a higher cumulative amount. Nothing touches the chain.

In [5]:
const channelId = depositPayload.payload.voucher.channelId;
const channelConfig = depositPayload.payload.channelConfig;
const voucher3 = await (async () => {
    await signVoucher(buyerSigner as any, channelId, (2n * PRICE).toString(), net.caip2 as any);
    return signVoucher(buyerSigner as any, channelId, (3n * PRICE).toString(), net.caip2 as any);
})();
console.log(`✅ 3 requests in one channel, cumulative ${fmt(voucher3.maxClaimableAmount)} — requests 2 and 3 cost no transaction`);

✅ 3 requests in one channel, cumulative 0.012 EURe — requests 2 and 3 cost no transaction


## 5. Seller approves EURe for the claim fee

The fee is charged in the channel's token, so a seller paid in EURe approves **EURe**: 1 EURe
(`10¹⁸`) covers 100 claims. This cell approves only if the allowance is short.

In [6]:
const ERC20 = [
    { name: "allowance", type: "function", stateMutability: "view", inputs: [{ name: "o", type: "address" }, { name: "s", type: "address" }], outputs: [{ name: "", type: "uint256" }] },
    { name: "approve", type: "function", stateMutability: "nonpayable", inputs: [{ name: "s", type: "address" }, { name: "v", type: "uint256" }], outputs: [{ name: "", type: "bool" }] },
] as const;
const allowance = await publicClient.readContract({ address: EURE, abi: ERC20, functionName: "allowance", args: [seller.address, facilitator] });
console.log(`🔍 seller → facilitator: ${fmt(allowance)} = ${allowance / FEE} claims`);

if (allowance < FEE && mayWrite) {
    const wallet = createWalletClient({ account: seller, chain: net.chain, transport: http(rpcUrl) });
    const hash = await wallet.writeContract({ address: EURE, abi: ERC20, functionName: "approve", args: [facilitator, BigInt(eureFee.recommended_amount)] });
    await publicClient.waitForTransactionReceipt({ hash });
    console.log(`✅ approved ${fmt(eureFee.recommended_amount)}: ${net.explorer}${hash}`);
} else if (allowance < FEE) {
    console.log("⏭️  approve skipped (DRY_RUN or unconfirmed mainnet)");
}

🔍 seller → facilitator: 0.99 EURe = 99 claims


## 6. Seller claims all three requests in one transaction

The seller's `receiverAuthorizer` signs the claim. The facilitator submits it, then pulls 0.01 EURe
from the seller (`10000000000000000` atomic).

In [7]:
import { claimBatchTypes, BATCH_SETTLEMENT_DOMAIN, BATCH_SETTLEMENT_ADDRESS } from "npm:@x402/evm@^2.26.0";

if (!depositSettled) {
    console.log("⏭️  no funded channel to claim from");
} else {
    const claimAuthorizerSignature = await receiverAuthorizer.signTypedData({
        domain: { ...BATCH_SETTLEMENT_DOMAIN, chainId: net.chain.id, verifyingContract: getAddress(BATCH_SETTLEMENT_ADDRESS) },
        types: claimBatchTypes, primaryType: "ClaimBatch",
        message: { claims: [{ channelId, maxClaimableAmount: BigInt(voucher3.maxClaimableAmount), totalClaimed: BigInt(voucher3.maxClaimableAmount) }] },
    });
    const claimPayload = {
        type: "claim",
        claims: [{
            voucher: { channel: channelConfig, maxClaimableAmount: voucher3.maxClaimableAmount },
            signature: voucher3.signature,
            totalClaimed: voucher3.maxClaimableAmount,
        }],
        claimAuthorizerSignature,
    };
    const claim = await post("/settle", {
        paymentPayload: { x402Version: 2, accepted: paymentRequirements, payload: claimPayload },
        paymentRequirements,
    });
    console.log("claim:", JSON.stringify(claim, null, 2));
    if (claim.success) {
        const info = claim.extensions?.facilitatorFees?.info;
        console.log(`✅ claimed ${fmt(voucher3.maxClaimableAmount)}: ${net.explorer}${claim.transaction}`);
        // On testnets, a seller listed in BATCH_SETTLEMENT_TEST_WALLETS is exempt from the fee:
        // the claim then carries no fee receipt at all.
        const exempt = !USE_MAINNET && (env.BATCH_SETTLEMENT_TEST_WALLETS ?? "").toLowerCase().includes(seller.address.toLowerCase());
        if (info) {
            console.log(`💸 fee ${fmt(info.facilitatorFeePaid)} ${BigInt(info.facilitatorFeePaid) === FEE ? "✅" : "❌"}, collection ${info.collection?.status}`);
        } else if (exempt) {
            console.log("ℹ️  no fee: the seller is in BATCH_SETTLEMENT_TEST_WALLETS (testnet exemption).");
            console.log("   Remove it from x402_facilitator/.env and restart the facilitator to test the EURe fee.");
        } else {
            console.log("❌ no fee receipt, and the seller is not exempt");
        }
    }
}

claim: {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x52774514e6459308ac6a7478f32b584131ddb43219a80a2ee14f8983f8125faf",
  "network": "eip155:8453",
  "fee": {
    "collected": true,
    "status": "collected",
    "txHash": "0xa0c8e567d9def5522ca067886ecf292446a0355de3dce4f16a4855b7df0454bc"
  },
  "extensions": {
    "facilitatorFees": {
      "info": {
        "version": "1",
        "facilitatorFeePaid": "10000000000000000",
        "asset": "eip155:8453/erc20:0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C",
        "model": "flat",
        "collection": {
          "status": "collected",
          "txHash": "0xa0c8e567d9def5522ca067886ecf292446a0355de3dce4f16a4855b7df0454bc"
        }
      }
    }
  }
}
✅ claimed 0.012 EURe: https://basescan.org/tx/0x52774514e6459308ac6a7478f32b584131ddb43219a80a2ee14f8983f8125faf
💸 fee 0.01 EURe ✅, collection collected
